# 04 - Analyse & Visualisation des résultats

Ce notebook génère toutes les visualisations pour l'article de blog.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.visualization import (
    plot_radar_chart,
    plot_heatmap,
    plot_latency_vs_quality,
    plot_bar_comparison,
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Charger les résultats

In [ ]:
# Charger le dernier fichier de résultats
results_dir = Path('../data/results')
csv_files = sorted(results_dir.glob('benchmark_results_*.csv'))

if csv_files:
    latest = csv_files[-1]
    print(f"Loading: {latest.name}")
    df = pd.read_csv(latest)
    df = df.dropna(subset=['faithfulness'])
    print(f"\n{len(df)} configurations evaluated successfully")
    display(df[['config_name', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'avg_latency']])
else:
    print("No results found! Run notebook 02 first.")

## 2. Générer toutes les visualisations

In [ ]:
# Radar chart
plot_radar_chart(df, '../data/results/radar_chart.png')

In [ ]:
# Heatmap
plot_heatmap(df, '../data/results/heatmap.png')

In [ ]:
# Latency vs Quality trade-off
plot_latency_vs_quality(df, '../data/results/latency_quality.png')

In [ ]:
# Bar comparison
plot_bar_comparison(df, '../data/results/bar_comparison.png')

## 3. Analyse des insights clés

In [ ]:
metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
df['avg_score'] = df[metrics].mean(axis=1)

print("="*60)
print("KEY INSIGHTS")
print("="*60)

best_overall = df.loc[df['avg_score'].idxmax()]
print(f"\n🏆 Meilleure config globale: {best_overall['config_name']} (score moyen: {best_overall['avg_score']:.4f})")

best_faithful = df.loc[df['faithfulness'].idxmax()]
print(f"\n📊 Meilleure faithfulness: {best_faithful['config_name']} ({best_faithful['faithfulness']:.4f})")

best_latency = df.loc[df['avg_latency'].idxmin()]
print(f"\n⚡ Plus rapide: {best_latency['config_name']} ({best_latency['avg_latency']:.2f}s)")

worst_overall = df.loc[df['avg_score'].idxmin()]
print(f"\n❌ Pire config: {worst_overall['config_name']} (score moyen: {worst_overall['avg_score']:.4f})")

print(f"\n\n--- Corrélation entre métriques ---")
corr = df[metrics].corr()
print(corr.to_string())

In [ ]:
# Matrice de corrélation visuelle
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Corrélation entre les métriques RAGAS')
plt.tight_layout()
plt.savefig('../data/results/correlation_matrix.png', dpi=150)
plt.show()

## 4. Conclusions pour l'article

Résumez ici vos conclusions principales basées sur vos résultats expérimentaux.

In [ ]:
print("""
CONCLUSIONS À INTÉGRER DANS L'ARTICLE :

1. Impact du chunking :
   - Comparer les scores des configs small/medium/large chunks
   - Le chunking sémantique apporte-t-il un gain significatif ?

2. Impact du retrieval :
   - Hybrid search vs naive : quel gain ?
   - Le reranking justifie-t-il son coût supplémentaire ?

3. Impact du LLM générateur :
   - GPT-4o vs GPT-4o-mini : le gain de qualité justifie-t-il le coût ?

4. Trade-off latence/qualité :
   - Quelle config offre le meilleur ratio ?

5. Limites de RAGAS :
   - Sensibilité au LLM-juge
   - Cas d'échec identifiés
   - Corrélation (ou non) avec l'évaluation humaine
""")